# NB17 — GPU stage: published baselines, Ar-APT external validation, positioning table

Stages run cheapest-and-highest-value first, each saving its own parquet immediately, with a
**time-budget guard** that skips later stages if the session nears your GPU limit.

- **A · Fast-DetectGPT** — analytic conditional-probability curvature (Bao et al. 2024), scored with
  AraGPT2, one forward pass per article. Includes a **sanity check on Ar-APT's balanced 400/400 set**:
  if the statistic is also at chance there, the failure is the method/surrogate (reportable); if it
  separates there, the problem is our corpus or the implementation.
- **B · Ar-APT** — both detectors on the external benchmark: unpolished human control (pure domain
  shift), AI-generated, and polished by level.
- **C · XLM-R 512** — the AraGenEval/AbjadGenEval family reimplemented on OUR corpus with their standard
  512 truncation (their data is not obtainable post-competition).
- **C2 · CAMeLBERT 512** — same 512 truncation, our encoder. This **disentangles** the two causes:
  C2 − C = the encoder's contribution; our K=9 model − C2 = the chunking contribution.
- **D · positioning table** — all rows with bootstrap CIs.

Everything printed; all artifacts parquet.

## 1 · Config

In [1]:
import os, re, time, json, math, glob, numpy as np, pandas as pd, torch
T_START = time.time()
TIME_BUDGET_MIN = 165

P_DATASET   = "/kaggle/input/datasets/bahaaqassem/aig-and-humang-dataset/dataset.parquet"
P_VSTAT16   = "/kaggle/input/datasets/bahaaqassem/aig-16-features/vstat16_scaled.parquet"
P_CHUNKS    = "/kaggle/input/notebooks/bahaaqassem/nb10-trackb-joint-finetune/chunks_K9.npz"                    # our corpus chunks (npz from NB10)
P_CK_HYBRID = "/kaggle/input/notebooks/bahaaqassem/nb10-trackb-joint-finetune/ckpt/last.pt"
P_CK_NEURAL = "/kaggle/input/notebooks/bahaaqassem/nb14-neural-only-production/ckpt_neural/last.pt"

# --- NB16 outputs (promote /kaggle/working of NB16 to a dataset and point here) ---
P_ARAPT     = "/kaggle/input/notebooks/bahaaqassem/nb16-prep-bootstrap/arapt_prepared.parquet"
P_ARAPT_V   = "/kaggle/input/notebooks/bahaaqassem/nb16-prep-bootstrap/arapt_vstat.parquet"
P_ARAPT_CH  = "/kaggle/input/notebooks/bahaaqassem/nb16-prep-bootstrap/arapt_chunks_K9.parquet"

MODEL_ID   = "CAMeL-Lab/bert-base-arabic-camelbert-msa"
SCORER_ID  = "aubmindlab/aragpt2-base"
XLMR_ID    = "xlm-roberta-base"
STAT_COLS  = ["burstiness","ttr","quote_ratio","function_word_ratio","compressibility"]
K_CHUNKS, MAX_CT, STRIDE = 9, 510, 460
HIDDEN, DROPOUT = 256, 0.0
FD_MAXLEN  = 1024
TRUNC_MAXLEN, TRUNC_EPOCHS, TRUNC_BS, TRUNC_LR = 512, 2, 8, 1e-5
SEEDS = [42, 43, 44]          # multi-seed: XLM-R collapsed once at 2e-5, so stability must be measured
N_BOOT, SEED = 2000, 42

RUN_A, RUN_B, RUN_C, RUN_C2 = True, True, True, True
np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"

def elapsed_min(): return (time.time()-T_START)/60
def budget_ok(need, tag):
    left = TIME_BUDGET_MIN - elapsed_min(); ok = left >= need
    print(f"[budget] {tag}: {elapsed_min():.1f} used, {left:.1f} left, needs ~{need} -> "
          f"{'RUN' if ok else 'SKIP'}", flush=True)
    return ok
print(f"[1/7] config loaded | device={DEV} | budget={TIME_BUDGET_MIN} min", flush=True)

WORK = "/kaggle/working"
def cached(fname):
    """find a previously-computed artifact in ANY attached input dataset (read-only)"""
    hits = glob.glob(f"/kaggle/input/**/{fname}", recursive=True)
    return hits[0] if hits else None
def outpath(fname):
    return os.path.join(WORK, fname)     # writes ALWAYS go to /kaggle/working

[1/7] config loaded | device=cuda | budget=165 min


## 2 · Load corpus, chunks, model class

In [2]:
df = pd.read_parquet(P_DATASET)
if "article_id" in df.columns: df = df.set_index("article_id")
v16 = pd.read_parquet(P_VSTAT16)
if "article_id" in v16.columns: v16 = v16.set_index("article_id")
df = df.loc[df.index.intersection(v16.index)]; v16 = v16.loc[df.index]
Xstat = v16[STAT_COLS].to_numpy(np.float32)
y     = df["label"].to_numpy(np.int64)
split = df["split"].to_numpy()
print(f"[2/7] corpus {len(df)} | train {int((split=='train').sum())} "
      f"val {int((split=='val').sum())} test {int((split=='test').sum())}", flush=True)

z = np.load(P_CHUNKS, allow_pickle=True); CH, NC = z["ch"], z["nch"]
assert CH.shape[0] == len(df)
print(f"[2/7] corpus chunks {CH.shape}", flush=True)

!pip install -q transformers
import torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
tok_c = AutoTokenizer.from_pretrained(MODEL_ID); PAD = tok_c.pad_token_id

class Net(nn.Module):
    def __init__(self, use_stat, stat_dim=5):
        super().__init__()
        self.use_stat = use_stat
        self.enc = AutoModel.from_pretrained(MODEL_ID)
        h = self.enc.config.hidden_size; fused = h + (stat_dim if use_stat else 0)
        self.register_buffer("mu", torch.zeros(fused)); self.register_buffer("sd", torch.ones(fused))
        self.head = nn.Sequential(nn.Linear(fused, HIDDEN), nn.LayerNorm(HIDDEN), nn.ReLU(),
                                  nn.Dropout(DROPOUT), nn.Linear(HIDDEN, 2))
    def encode(self, ids, nch):
        B,K,L = ids.shape; flat = ids.view(B*K, L); att = (flat != PAD).long()
        cls = self.enc(input_ids=flat, attention_mask=att).last_hidden_state[:,0,:].view(B,K,-1)
        m = (torch.arange(K, device=ids.device)[None,:] < nch[:,None]).float().unsqueeze(-1)
        return (cls*m).sum(1)/m.sum(1).clamp(min=1)
    def forward(self, ids, nch, stat=None):
        v = self.encode(ids, nch)
        if self.use_stat: v = torch.cat([v, stat], 1)
        v = (v - self.mu)/self.sd
        return self.head(v)

class DS(Dataset):
    def __init__(self, CH, NC, X): self.CH, self.NC, self.X = CH, NC, X
    def __len__(self): return len(self.CH)
    def __getitem__(self, i):
        return (torch.from_numpy(self.CH[i].astype(np.int64)), int(self.NC[i]),
                torch.from_numpy(self.X[i]))
def collate(b):
    return (torch.stack([x[0] for x in b]), torch.tensor([x[1] for x in b]),
            torch.stack([x[2] for x in b]))

@torch.no_grad()
def predict(model, CHx, NCx, Xx, tag):
    model.eval(); P=[]; t0=time.time()
    for bi,(ids,nch,st) in enumerate(DataLoader(DS(CHx,NCx,Xx), batch_size=8, collate_fn=collate), 1):
        ids,nch,st = ids.to(DEV), nch.to(DEV), st.to(DEV)
        with torch.amp.autocast('cuda'):
            P.append(torch.softmax(model(ids,nch,st),1)[:,1].float().cpu().numpy())
        if bi % 40 == 0: print(f"        {tag} {bi*8}/{len(CHx)} ({time.time()-t0:.0f}s)", flush=True)
    return np.concatenate(P)
print("[2/7] model class + predictor ready", flush=True)

[2/7] corpus 7101 | train 5363 val 645 test 1093
[2/7] corpus chunks (7101, 9, 512)


config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[2/7] model class + predictor ready


## 3 · Stage A — Fast-DetectGPT (+ sanity check on Ar-APT)

`d(x) = (log p(x) − Σμ_j) / sqrt(Σσ²_j)` where `μ_j`, `σ²_j` are the mean and variance of the log-prob
the scoring model would assign to a token drawn from its own distribution at position j. Higher d =
more machine-like. Analytic, so one forward pass and no perturbation model.

**Caveat for the thesis:** the method is designed white-box. Our generators are closed, so AraGPT2 is a
*surrogate* — a black-box setting the paper reports as substantially weaker. Any gap to our supervised
model measures the value of supervision, not a like-for-like defeat.

In [3]:
FD_OUT = outpath("nb17_fastdetectgpt.parquet")
FD_CACHE = cached("nb17_fastdetectgpt.parquet")
FD = None
if FD_CACHE:
    FD = pd.read_parquet(FD_CACHE)
    print(f"[3/7] loaded cached scores from {FD_CACHE} (skips ~6 min)", flush=True)
elif RUN_A and budget_ok(30, "A Fast-DetectGPT"):
    from transformers import AutoModelForCausalLM
    tok_s = AutoTokenizer.from_pretrained(SCORER_ID)
    scorer = AutoModelForCausalLM.from_pretrained(SCORER_ID).to(DEV).eval()
    print(f"[3/7] scorer {SCORER_ID} ({sum(p.numel() for p in scorer.parameters())/1e6:.0f}M)", flush=True)

    @torch.no_grad()
    def fastdetect(text):
        ids = tok_s(str(text), return_tensors="pt", truncation=True, max_length=FD_MAXLEN).input_ids.to(DEV)
        if ids.shape[1] < 10: return float("nan")
        with torch.amp.autocast('cuda'):
            logits = scorer(ids).logits[:, :-1, :].float()
        labels = ids[:, 1:]
        lsm = F.log_softmax(logits, dim=-1)
        lp_obs = lsm.gather(-1, labels.unsqueeze(-1)).squeeze(-1).sum().item()
        p = lsm.exp()
        mu_j  = (p * lsm).sum(-1)
        var_j = (p * lsm.pow(2)).sum(-1) - mu_j.pow(2)
        mu, sd = mu_j.sum().item(), math.sqrt(max(var_j.sum().item(), 1e-9))
        return (lp_obs - mu) / sd

    t0=time.time(); scores=[]
    for k, t in enumerate(df["text"].tolist(), 1):
        scores.append(fastdetect(t))
        if k % 500 == 0: print(f"[3/7]   scored {k}/{len(df)} ({time.time()-t0:.0f}s)", flush=True)
    FD = pd.DataFrame({"article_id": df.index.values, "label": y, "split": split,
                       "fastdetect": np.array(scores, np.float32)})
    FD.to_parquet(FD_OUT, index=False)
    print(f"[3/7] scored our corpus in {(time.time()-t0)/60:.1f} min -> {FD_OUT}", flush=True)

    # ---- SANITY CHECK on Ar-APT's balanced unpolished set -------------------
    try:
        ARs = pd.read_parquet(P_ARAPT)
        sub = ARs[ARs.kind.isin(["human_orig","ai_generated"])].reset_index(drop=True)
        print(f"[3/7] sanity check on Ar-APT balanced set (n={len(sub)}) ...", flush=True)
        sc = np.array([fastdetect(t) for t in sub.text.tolist()], np.float32)
        ok = ~np.isnan(sc)
        a = 100*roc_auc_score(sub.label.to_numpy()[ok], sc[ok])
        print(f"[3/7]   Ar-APT AUC {a:.2f} | human {sc[ok & (sub.label.to_numpy()==0)].mean():.3f} "
              f"| AI {sc[ok & (sub.label.to_numpy()==1)].mean():.3f}", flush=True)
        print("[3/7]   ~50 here too => surrogate/method failure (reportable). "
              "Separates here => the issue is our corpus or the implementation.", flush=True)
        pd.DataFrame({"arapt_id": sub.arapt_id, "label": sub.label, "fastdetect": sc}) \
          .to_parquet("/kaggle/working/nb17_fastdetectgpt_arapt.parquet", index=False)
    except Exception as e:
        print(f"[3/7]   sanity check skipped: {e}", flush=True)
    del scorer; torch.cuda.empty_cache()
else:
    print("[3/7] stage A skipped", flush=True)

FD_THR = None
if FD is not None:
    f = FD.dropna(subset=["fastdetect"]); va, te = f[f.split=="val"], f[f.split=="test"]
    ths = np.quantile(va.fastdetect, np.linspace(0.01, 0.99, 99))
    FD_THR = max(ths, key=lambda t: f1_score(va.label,(va.fastdetect>=t).astype(int),average="macro"))
    pr = (te.fastdetect >= FD_THR).astype(int)
    print(f"[3/7] Fast-DetectGPT on OUR TEST: AUC {100*roc_auc_score(te.label,te.fastdetect):.2f} | "
          f"MacroF1 {100*f1_score(te.label,pr,average='macro'):.2f} | "
          f"Acc {100*accuracy_score(te.label,pr):.2f}  (threshold from VAL)", flush=True)
    print(f"[3/7]   score means: human {f[f.label==0].fastdetect.mean():.3f} | "
          f"AI {f[f.label==1].fastdetect.mean():.3f}", flush=True)

[3/7] loaded cached scores from /kaggle/input/datasets/bahaaqassem/nb17-dataset/nb17_fastdetectgpt.parquet (skips ~6 min)
[3/7] Fast-DetectGPT on OUR TEST: AUC 50.44 | MacroF1 49.38 | Acc 52.33  (threshold from VAL)
[3/7]   score means: human -5.085 | AI -3.756


## 4 · Stage B — Ar-APT external validation

In [4]:
AR_OUT = outpath("nb17_arapt_preds.parquet")
AR_CACHE = cached("nb17_arapt_preds.parquet")
AR = None
if AR_CACHE:
    AR = pd.read_parquet(AR_CACHE)
    print(f"[4/7] loaded cached Ar-APT predictions from {AR_CACHE} (skips ~7 min)", flush=True)
elif RUN_B and budget_ok(25, "B Ar-APT"):
    AR  = pd.read_parquet(P_ARAPT)
    ARV = pd.read_parquet(P_ARAPT_V)
    ARC = pd.read_parquet(P_ARAPT_CH)
    AR  = AR.set_index("arapt_id").loc[ARV.arapt_id].reset_index()
    ARC = ARC.set_index("arapt_id").loc[AR.arapt_id].reset_index()
    Xa  = ARV[STAT_COLS].to_numpy(np.float32)
    shape = tuple(ARC.chunk_shape.iloc[0])
    CHa = np.array(ARC.chunks.tolist(), dtype=np.int32).reshape(len(ARC), *shape)
    NCa = ARC.n_chunks.to_numpy()
    print(f"[4/7] Ar-APT {len(AR)} | chunks {CHa.shape} | {AR.kind.value_counts().to_dict()}", flush=True)
    for name, ck, use_stat in [("hybrid", P_CK_HYBRID, True), ("neural", P_CK_NEURAL, False)]:
        m = Net(use_stat).to(DEV)
        m.load_state_dict(torch.load(ck, map_location=DEV, weights_only=False)["model"])
        print(f"[4/7] {name} loaded (fused {m.mu.numel()})", flush=True)
        AR[f"p_{name}"] = predict(m, CHa, NCa, Xa, f"{name}/arapt")
        del m; torch.cuda.empty_cache()
    AR.drop(columns=["text"]).to_parquet(AR_OUT, index=False)
    print(f"[4/7] saved {AR_OUT}", flush=True)
else:
    print("[4/7] stage B skipped", flush=True)

if AR is not None:
    def rate(p): return 100.0*float((np.asarray(p) >= .5).mean())
    print("\n[4/7] DOMAIN-TRANSFER CONTROL (unpolished):", flush=True)
    ho = AR[AR.kind=="human_orig"]; ai = AR[AR.kind=="ai_generated"]
    print(f"[4/7]   human_orig   n={len(ho):4d}  FPR hybrid {rate(ho.p_hybrid):6.2f}  neural {rate(ho.p_neural):6.2f}", flush=True)
    print(f"[4/7]   ai_generated n={len(ai):4d}  TPR hybrid {rate(ai.p_hybrid):6.2f}  neural {rate(ai.p_neural):6.2f}", flush=True)
    if len(ho) and len(ai):
        for nm in ("hybrid","neural"):
            acc = ((1-rate(ho[f"p_{nm}"])/100)*len(ho) + rate(ai[f"p_{nm}"])/100*len(ai))/(len(ho)+len(ai))*100
            print(f"[4/7]   BALANCED-SET ACCURACY {nm:6s} {acc:.2f}%  "
                  f"(zero training on this domain — the like-for-like number vs published detectors)", flush=True)
    print("\n[4/7] POLISHING (human articles -> this is FPR; lower is better):", flush=True)
    pol = AR[AR.kind=="human_polished"]; rows=[]
    for lvl in sorted(pol.level.dropna().unique()):
        s = pol[pol.level==lvl]
        rows.append({"level":int(lvl), "n":len(s),
                     "FPR_hyb":round(rate(s.p_hybrid),2), "FPR_neu":round(rate(s.p_neural),2)})
    T = pd.DataFrame(rows); T["gap"]=(T.FPR_hyb-T.FPR_neu).round(2)
    print(T.to_string(index=False), flush=True)
    print("\n[4/7] by polishing model:", flush=True)
    print(pol.groupby("polish_model").apply(lambda s: pd.Series({
        "n":len(s), "FPR_hyb":round(rate(s.p_hybrid),2), "FPR_neu":round(rate(s.p_neural),2)}),
        include_groups=False).to_string(), flush=True)
    print("\n[4/7] Ar-APT reported Originality.AI 92 -> 12 and Claude-4 Sonnet 83.5 -> 57.6 under", flush=True)
    print("[4/7] polishing. Only the balanced-set accuracy above is directly comparable to those.", flush=True)

[4/7] loaded cached Ar-APT predictions from /kaggle/input/datasets/bahaaqassem/nb17-dataset/nb17_arapt_preds.parquet (skips ~7 min)

[4/7] DOMAIN-TRANSFER CONTROL (unpolished):
[4/7]   human_orig   n= 400  FPR hybrid  12.00  neural  19.25
[4/7]   ai_generated n= 400  TPR hybrid  72.00  neural  81.50
[4/7]   BALANCED-SET ACCURACY hybrid 80.00%  (zero training on this domain — the like-for-like number vs published detectors)
[4/7]   BALANCED-SET ACCURACY neural 81.12%  (zero training on this domain — the like-for-like number vs published detectors)

[4/7] POLISHING (human articles -> this is FPR; lower is better):
 level   n  FPR_hyb  FPR_neu    gap
    10 496    16.33    28.02 -11.69
    25 449    16.04    27.17 -11.13
    50 494    15.18    28.34 -13.16
    75 497    22.74    34.00 -11.26

[4/7] by polishing model:
                     n  FPR_hyb  FPR_neu
polish_model                            
DS               397.0    20.65    30.98
GPT 4            196.0    15.31    28.57
GPT4     

## 5 · Stages C and C2 — 512-truncation models (encoder vs chunking, disentangled)

One routine, run twice. **C** = XLM-R (the competition family). **C2** = CAMeLBERT (our encoder) under the
*same* 512 truncation. Then:

- `C2 − C` = the **encoder** contribution
- `our K=9 model − C2` = the **chunking** contribution

Without C2 those two causes are confounded, which is exactly the objection an examiner would raise.

In [5]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.utils.class_weight import compute_class_weight

def train_trunc(model_id, out_path, tag, seed=42):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); np.random.seed(seed)
    tk = AutoTokenizer.from_pretrained(model_id)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2).to(DEV)
    print(f"[5/7] {tag}: {model_id} | truncation {TRUNC_MAXLEN} | lr {TRUNC_LR:g} | seed {seed}", flush=True)
    enc = tk(df["text"].astype(str).tolist(), truncation=True, max_length=TRUNC_MAXLEN,
             padding="max_length", return_tensors="np")
    IDS, ATT = enc["input_ids"].astype(np.int32), enc["attention_mask"].astype(np.int8)
    print(f"[5/7] {tag}: tokenised {IDS.shape}", flush=True)

    class XDS(Dataset):
        def __init__(self, rows): self.rows = rows
        def __len__(self): return len(self.rows)
        def __getitem__(self, k):
            i = self.rows[k]
            return (torch.tensor(IDS[i], dtype=torch.long),
                    torch.tensor(ATT[i], dtype=torch.long), int(y[i]))
    def xcol(b):
        return (torch.stack([x[0] for x in b]), torch.stack([x[1] for x in b]),
                torch.tensor([x[2] for x in b]))

    tr = [i for i in range(len(df)) if split[i]=="train"]
    te = [i for i in range(len(df)) if split[i]=="test"]
    cw = compute_class_weight("balanced", classes=np.array([0,1]), y=y[tr])
    lossf = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32, device=DEV))
    dl = DataLoader(XDS(tr), batch_size=TRUNC_BS, shuffle=True, collate_fn=xcol, drop_last=True)
    total = len(dl)*TRUNC_EPOCHS
    opt = torch.optim.AdamW(mdl.parameters(), lr=TRUNC_LR, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.06*total), total)
    scaler = torch.amp.GradScaler('cuda')
    print(f"[5/7] {tag}: {len(tr)} rows, {len(dl)} batches/epoch, {total} steps", flush=True)

    t0=time.time(); step=0; mdl.train()
    for ep in range(TRUNC_EPOCHS):
        for ids,att,yy in dl:
            ids,att,yy = ids.to(DEV), att.to(DEV), yy.to(DEV)
            with torch.amp.autocast('cuda'):
                loss = lossf(mdl(input_ids=ids, attention_mask=att).logits, yy)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
            step += 1
            if step % 100 == 0:
                print(f"[5/7] {tag}: ep{ep+1}/{TRUNC_EPOCHS} step {step}/{total} "
                      f"loss {loss.item():.4f} | {(time.time()-t0)/60:.1f}m", flush=True)
        print(f"[5/7] {tag}: epoch {ep+1} done ({(time.time()-t0)/60:.1f}m)", flush=True)

    mdl.eval(); P=[]
    with torch.no_grad():
        for ids,att,yy in DataLoader(XDS(te), batch_size=16, collate_fn=xcol):
            ids,att = ids.to(DEV), att.to(DEV)
            with torch.amp.autocast('cuda'):
                P.append(torch.softmax(mdl(input_ids=ids, attention_mask=att).logits,1)[:,1].float().cpu().numpy())
    out = pd.DataFrame({"article_id": df.index.values[te], "label": y[te], "p": np.concatenate(P)})
    out.to_parquet(out_path, index=False)
    pr = (out.p>=.5).astype(int)
    print(f"[5/7] {tag} TEST: MacroF1 {100*f1_score(out.label,pr,average='macro'):.2f} | "
          f"Acc {100*accuracy_score(out.label,pr):.2f} | AUC {100*roc_auc_score(out.label,out.p):.2f}", flush=True)
    del mdl; torch.cuda.empty_cache()
    return out

def run_seeds(model_id, tag, prefix, need=10):
    res = []
    for sd in SEEDS:
        fname = f"{prefix}_lr{TRUNC_LR:g}_seed{sd}.parquet"   # LR in the name invalidates old-LR caches
        p, hit = outpath(fname), cached(fname)
        if hit:
            o = pd.read_parquet(hit); print(f"[5/7] {tag} seed {sd}: cached ({hit})", flush=True)
        elif budget_ok(need, f"{tag} seed {sd}"):
            o = train_trunc(model_id, p, f"{tag} s{sd}", seed=sd)
        else:
            print(f"[5/7] {tag} seed {sd}: SKIPPED (budget)", flush=True); break
        pr = (o.p >= .5).astype(int)
        res.append({"seed": sd, "preds": o,
                    "f1":  100*f1_score(o.label, pr, average="macro"),
                    "auc": 100*roc_auc_score(o.label, o.p)})
    if res:
        f1s = np.array([r["f1"] for r in res])
        sd_ = f1s.std(ddof=1) if len(f1s) > 1 else 0.0
        print(f"[5/7] {tag} ACROSS {len(res)} SEEDS: MacroF1 mean {f1s.mean():.2f} sd {sd_:.2f} "
              f"min {f1s.min():.2f} max {f1s.max():.2f}", flush=True)
        for r in res:
            print(f"[5/7]     seed {r['seed']}: F1 {r['f1']:.2f}  AUC {r['auc']:.2f}", flush=True)
    return res

XL_RES = run_seeds(XLMR_ID,  "XLM-R 512",     "nb17_xlmr512")      if RUN_C  else []
CB_RES = run_seeds(MODEL_ID, "CAMeLBERT 512", "nb17_camelbert512") if RUN_C2 else []

[budget] XLM-R 512 seed 42: 0.4 used, 164.6 left, needs ~10 -> RUN


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; n

[5/7] CAMeLBERT 512 s44: CAMeL-Lab/bert-base-arabic-camelbert-msa | truncation 512 | lr 1e-05 | seed 44
[5/7] CAMeLBERT 512 s44: tokenised (7101, 512)
[5/7] CAMeLBERT 512 s44: 5363 rows, 670 batches/epoch, 1340 steps


/tmp/ipykernel_22/577782319.py:42: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()


[5/7] CAMeLBERT 512 s44: ep1/2 step 100/1340 loss 0.0702 | 0.4m
[5/7] CAMeLBERT 512 s44: ep1/2 step 200/1340 loss 0.0345 | 0.7m
[5/7] CAMeLBERT 512 s44: ep1/2 step 300/1340 loss 0.0095 | 1.1m
[5/7] CAMeLBERT 512 s44: ep1/2 step 400/1340 loss 0.0274 | 1.4m
[5/7] CAMeLBERT 512 s44: ep1/2 step 500/1340 loss 0.0039 | 1.8m
[5/7] CAMeLBERT 512 s44: ep1/2 step 600/1340 loss 0.0024 | 2.2m
[5/7] CAMeLBERT 512 s44: epoch 1 done (2.4m)
[5/7] CAMeLBERT 512 s44: ep2/2 step 700/1340 loss 0.0086 | 2.5m
[5/7] CAMeLBERT 512 s44: ep2/2 step 800/1340 loss 0.0107 | 2.9m
[5/7] CAMeLBERT 512 s44: ep2/2 step 900/1340 loss 0.0023 | 3.2m
[5/7] CAMeLBERT 512 s44: ep2/2 step 1000/1340 loss 0.0040 | 3.6m
[5/7] CAMeLBERT 512 s44: ep2/2 step 1100/1340 loss 0.0097 | 4.0m
[5/7] CAMeLBERT 512 s44: ep2/2 step 1200/1340 loss 0.0017 | 4.3m
[5/7] CAMeLBERT 512 s44: ep2/2 step 1300/1340 loss 0.0016 | 4.7m
[5/7] CAMeLBERT 512 s44: epoch 2 done (4.8m)
[5/7] CAMeLBERT 512 s44 TEST: MacroF1 99.08 | Acc 99.09 | AUC 100.00
[5/7]

## 6 · Stage D — positioning table with bootstrap CIs

In [6]:
rows = []
def boot_f1(yt, pred, B=400):
    yt, pred = np.asarray(yt), np.asarray(pred)
    idx = np.random.randint(0, len(yt), size=(B, len(yt)))
    v = np.array([100*f1_score(yt[i], pred[i], average="macro") for i in idx])
    return round(float(np.percentile(v,2.5)),2), round(float(np.percentile(v,97.5)),2)

def add(name, kind, yt, p, thr=0.5):
    pred = (np.asarray(p) >= thr).astype(int)
    f1 = 100*f1_score(yt, pred, average="macro")
    try: auc = 100*roc_auc_score(yt, p)
    except Exception: auc = float("nan")
    lo, hi = boot_f1(yt, pred)
    rows.append({"system":name, "type":kind, "MacroF1":round(f1,2),
                 "F1_lo":lo, "F1_hi":hi, "AUC":round(auc,2), "n":len(yt)})
    print(f"[6/7]   {name:34s} F1 {f1:6.2f} [{lo:6.2f},{hi:6.2f}]  AUC {auc:6.2f}", flush=True)

print("[6/7] POSITIONING TABLE (our test split)", flush=True)
if FD is not None and FD_THR is not None:
    f = FD.dropna(subset=["fastdetect"]); te = f[f.split=="test"]
    add("Fast-DetectGPT (zero-shot)", "zero-shot", te.label.to_numpy(), te.fastdetect.to_numpy(), FD_THR)
def add_multiseed(name, res):
    if not res: return None
    f1s = np.array([r["f1"] for r in res]); aucs = np.array([r["auc"] for r in res])
    med = res[int(np.argsort(f1s)[len(f1s)//2])]          # median-seed run
    rows.append({"system": name, "type": "supervised", "MacroF1": round(float(f1s.mean()),2),
                 "F1_lo": round(float(f1s.min()),2), "F1_hi": round(float(f1s.max()),2),
                 "AUC": round(float(aucs.mean()),2), "n": len(med["preds"])})
    print(f"[6/7]   {name:34s} F1 {f1s.mean():6.2f} [{f1s.min():6.2f},{f1s.max():6.2f}] "
          f"AUC {aucs.mean():6.2f}  ({len(res)} seeds)", flush=True)
    return float(f1s.mean())

xl_mean = add_multiseed("XLM-R 512 (competition-style)", XL_RES)
cb_mean = add_multiseed("CAMeLBERT 512 (truncation)",    CB_RES)

cand = glob.glob("/kaggle/input/notebooks/bahaaqassem/nb15-stress-evaluation/nb15_stress_results.parquet", recursive=True)
print(f"[6/7] NB15 stress results found at: {cand}", flush=True)
if cand:
    S = pd.read_parquet(cand[0])
    lab = df.loc[S.article_id, "label"].to_numpy()
    add("CAMeLBERT neural K=9 (ours)", "supervised", lab, S.p_neural_base.to_numpy())
    add("Hybrid 773 K=9 (ours)",       "supervised", lab, S.p_hybrid_base.to_numpy())
else:
    print("[6/7] !! NB15 results not found — add that dataset to get the final two rows", flush=True)

TAB = pd.DataFrame(rows)
TAB.to_parquet("/kaggle/working/nb17_positioning_table.parquet", index=False)
print("\n[6/7] final table:\n" + TAB.to_string(index=False), flush=True)

def get(nm):
    r = TAB[TAB.system.str.startswith(nm)]
    return float(r.MacroF1.iloc[0]) if len(r) else None
x, c, n = xl_mean, cb_mean, get("CAMeLBERT neural K=9")
print("\n[7/7] DISENTANGLING encoder vs chunking:", flush=True)
if x is not None and c is not None:
    print(f"[7/7]   encoder  (CAMeLBERT512 - XLMR512)   = {c-x:+.2f} pp", flush=True)
if c is not None and n is not None:
    print(f"[7/7]   chunking (K=9 - CAMeLBERT512)       = {n-c:+.2f} pp", flush=True)
print(f"\n[7/7] TOTAL SESSION {elapsed_min():.1f} min", flush=True)
print("[7/7] NOTE: the zero-shot row is not a like-for-like defeat — untrained on this corpus and", flush=True)
print("[7/7]       using a surrogate scorer. It measures the value of supervision.", flush=True)

[6/7] POSITIONING TABLE (our test split)
[6/7]   Fast-DetectGPT (zero-shot)         F1  49.38 [ 46.22, 52.29]  AUC  50.44
[6/7]   XLM-R 512 (competition-style)      F1  98.38 [ 97.99, 98.99] AUC  99.99  (3 seeds)
[6/7]   CAMeLBERT 512 (truncation)         F1  98.75 [ 97.98, 99.18] AUC 100.00  (3 seeds)
[6/7] NB15 stress results found at: ['/kaggle/input/notebooks/bahaaqassem/nb15-stress-evaluation/nb15_stress_results.parquet']
[6/7]   CAMeLBERT neural K=9 (ours)        F1  99.88 [ 99.63,100.00]  AUC 100.00
[6/7]   Hybrid 773 K=9 (ours)              F1  99.76 [ 99.27,100.00]  AUC 100.00

[6/7] final table:
                       system       type  MacroF1  F1_lo  F1_hi    AUC    n
   Fast-DetectGPT (zero-shot)  zero-shot    49.38  46.22  52.29  50.44 1093
XLM-R 512 (competition-style) supervised    98.38  97.99  98.99  99.99 1093
   CAMeLBERT 512 (truncation) supervised    98.75  97.98  99.18 100.00 1093
  CAMeLBERT neural K=9 (ours) supervised    99.88  99.63 100.00 100.00  820
       